In [1]:
from dotenv import load_dotenv
import os

# .envファイルの絶対パスを指定
env_path = os.path.join("..", ".env")  # 親ディレクトリの.envを指定
load_dotenv(env_path)

True

In [2]:
base_url = os.getenv("XR_SERVER_API")

In [3]:
from enum import Enum


class FurnitureType(Enum):
    TABLE="TABLE"
    TV="TV"
    

In [4]:
import httpx
import json

data = {
    "isInFov" : True, 
    "order": "proximity", 
    "range": 0, 
    "furnitureType" : FurnitureType.TABLE.value
}

res = httpx.post(base_url+"/furniture/fov", json = data)
data = json.load(res)
data


{'status': 'success',
 'furnitures': ['{"position":{"x":-0.4238522,"y":-0.667920232,"z":4.70446253},"distance_from_user":4.770507,"id":"3866dc8d-1e08-4f32-84f0-2877e65c6a2f","name":"テーブル","FurnitureType":"TABLE"}']}

In [5]:
data = {
    "direction" : "FRONT", 
    "order": "proximity", 
    "range": 0, 
    "furnitureType" : FurnitureType.TABLE.value
}


res = httpx.post(base_url + "/furniture/direction", json=data)
data = res.json()  # こっちでOK（json.load(res)はファイル用）
furnitures = [json.loads(f) for f in data["furnitures"]]  # JSON文字列 → dictに変換

print(data)
print()
print(furnitures[0]["id"])


{'status': 'success', 'furnitures': ['{"position":{"x":-0.4238522,"y":-0.667920232,"z":4.70446253},"distance_from_user":4.770507,"id":"3866dc8d-1e08-4f32-84f0-2877e65c6a2f","name":"テーブル","FurnitureType":"TABLE"}']}

3866dc8d-1e08-4f32-84f0-2877e65c6a2f


In [6]:

sending = {
    "direction" : "FRONT", 
    "order": "proximity", 
    "range": 0, 
    "furnitureType" : FurnitureType.TABLE.value
}

res = httpx.post(base_url + "/furniture/direction", json=sending)
data = res.json()  # こっちでOK（json.load(res)はファイル用）
furnitures = [json.loads(f) for f in data["furnitures"]] # JSON文字列 → dictに変換

print(f"RECEIVED FURNITUR: name {furnitures[0]['name']}, id {furnitures[0]['id']}")



sending = {
    "id" : furnitures[0]['id'], 
    "order": "proximity", 
    "range": 0
}

print("sending", sending)
res = httpx.post(base_url + "/furniture/find_device_with_furniture", json=sending)
data = res.json()  # こっちでOK（json.load(res)はファイル用）
data

RECEIVED FURNITUR: name テーブル, id 3866dc8d-1e08-4f32-84f0-2877e65c6a2f
sending {'id': '3866dc8d-1e08-4f32-84f0-2877e65c6a2f', 'order': 'proximity', 'range': 0}


{'status': 'success',
 'devices': [{'id': '4854e249-50d2-4da2-af10-e373f751fe0c',
   'name': 'フロアライト',
   'position': {'x': 0.725365639, 'y': 0.768305957, 'z': -0.06326866},
   'distance_from_user': 4.652061,
   'angle': 14.6652651}]}

In [7]:
import httpx 

In [11]:
sending = {
   
    "range": 5, 
    "furnitureType" : FurnitureType.TABLE.value
}

res = httpx.post(base_url + "/furniture/get", json=sending)
data = res.json() 

data

{'status': 'success',
 'devices': [{'id': '9f8a9c29-dcee-4a56-96f0-d4b3e2383a87',
   'name': 'フロアライト',
   'position': {'x': 0.725365639, 'y': 0.768305957, 'z': -0.06326866},
   'distance_from_user': 4.652061,
   'angle': 14.6652651},
  {'id': 'c18eaafd-021d-4165-bd18-1b5883a195fb',
   'name': 'フロアライト (1)',
   'position': {'x': -0.156467974, 'y': 0.767130733, 'z': -0.0359420776},
   'distance_from_user': 4.705497,
   'angle': -0.985561967}]}

In [36]:
from langchain.tools import tool
from typing import Annotated
import httpx
@tool
def getFurnitureDevice(
    furniture_type: Annotated[str, """"TV | TABLE"""],
    range: Annotated[float, "distance or range. Default is 5 if not specified"]
):
    """"You can use this when you need to find devices around a furniture"""
    sending = {
   
    "range": 5, 
    "furnitureType" : furniture_type
        }


    res = httpx.post(f"{base_url}/furniture/get", json=sending)
    return res.json()

{'status': 'success',
 'devices': [{'id': 'bc574027-b293-40ea-ae55-6c3d7f11f921',
   'name': 'フロアライト',
   'position': {'x': 0.725365639, 'y': 0.768305957, 'z': -0.06326866},
   'distance_from_user': 4.652061,
   'angle': 14.6652651},
  {'id': '40790558-59bc-4306-9216-2c9e9ef43d4b',
   'name': 'フロアライト (1)',
   'position': {'x': -0.156467974, 'y': 0.767130733, 'z': -0.0359420776},
   'distance_from_user': 4.705497,
   'angle': -0.985561967}]}

In [39]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
llm_with_tool = llm.bind_tools(tools=[getFurnitureDevice],strict=True)



In [40]:
llm_with_tool.invoke("Could you please find the lights on the table")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}